# Run Direct in WSL

## Check Dir

In [1]:
import os
os.makedirs(os.path.expanduser('~/DPCC'), exist_ok=True)
print("✓ Working directory ready")

✓ Working directory ready


# Register the Kernel - Change the ipynb Kernel into dpcc

In [ ]:
# 1. Install the bridge into the dpcc environment
~/miniconda3/envs/dpcc/bin/pip install ipykernel -q

# 2. Now register the kernel
~/miniconda3/envs/dpcc/bin/python -m ipykernel install --user --name dpcc --display-name "Python 3.10 (dpcc)"

## Install Miniconda into Colab (to get Python 3.10)

In [ ]:
%%bash
if [ ! -f "$HOME/miniconda.sh" ]; then
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh \
        -O ~/miniconda.sh
fi
bash ~/miniconda.sh -b -p ~/miniconda3 -u
echo "✓ Miniconda installed"
~/miniconda3/bin/conda --version

## Clone the DPCC Repo

In [ ]:
%%bash
cd ~/DPCC

if [ ! -d "dpcc" ]; then
    git clone --recurse-submodules "https://github.com/ghubliming/dpcc.git"
    echo "Cloned successfully."
else
    echo "Repo already exists, pulling latest..."
    cd dpcc && git pull
fi

## Clone the D3IL

In [ ]:
# ── Check if d3il exists and is populated, clone if not ──────
D3IL="$HOME/DPCC/dpcc/d3il"
PIP="$HOME/miniconda3/envs/dpcc/bin/pip"

echo "=== Checking d3il ==="
if [ ! -f "$D3IL/environments/d3il/setup.py" ]; then
    echo "d3il missing or empty — cloning from ALRhub..."
    rm -rf "$D3IL"
    git clone https://github.com/ALRhub/d3il.git "$D3IL"
    echo "✓ d3il cloned"
else
    echo "✓ d3il already exists, skipping clone"
fi

echo ""
echo "=== Verifying ==="
ls "$D3IL/environments/d3il/setup.py" \
    && echo "✓ d3il core setup.py found" \
    || { echo "✗ setup.py MISSING — clone may have failed"; exit 1; }

ls "$D3IL/environments/d3il/envs/gym_avoiding_env/setup.py" \
    && echo "✓ gym_avoiding_env setup.py found" \
    || { echo "✗ gym_avoiding_env setup.py MISSING"; exit 1; }

echo ""
echo "=== Installing D3IL ==="
$PIP install -e "$D3IL/environments/d3il" -q \
    && echo "✓ d3il core installed" \
    || { echo "✗ d3il core FAILED"; exit 1; }

$PIP install -e "$D3IL/environments/d3il/envs/gym_avoiding_env" -q \
    && echo "✓ gym_avoiding_env installed" \
    || { echo "✗ gym_avoiding_env FAILED"; exit 1; }

echo ""
echo "=== Done — D3IL ready ==="

## Create conda env with Python 3.10

In [ ]:
%%bash
~/miniconda3/bin/conda tos accept --override-channels \
    --channel https://repo.anaconda.com/pkgs/main
~/miniconda3/bin/conda tos accept --override-channels \
    --channel https://repo.anaconda.com/pkgs/r

~/miniconda3/bin/conda create -n dpcc python=3.10 -y -q
echo "✓ conda env 'dpcc' created with Python 3.10"
~/miniconda3/envs/dpcc/bin/python --version

## Install PyTorch (CUDA124)

In [ ]:
%%bash
~/miniconda3/envs/dpcc/bin/pip install \
    torch torchvision \
    --index-url https://download.pytorch.org/whl/cpu -q
echo "✓ PyTorch (CPU) installed"
~/miniconda3/envs/dpcc/bin/python -c \
    "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.cuda.is_available())"

## Install All Requirements

In [ ]:
%%bash
PIP="$HOME/miniconda3/envs/dpcc/bin/pip"
cd ~/DPCC/dpcc

echo "========================================="
echo " INSTALLING FROM requirements.txt"
echo "========================================="
echo "Using pip: $($PIP --version)"
echo ""

if [ ! -f requirements.txt ]; then
  echo "ERROR: requirements.txt not found in $(pwd)"
  exit 1
fi

echo "Found requirements.txt -- $(wc -l < requirements.txt) packages listed"
echo ""

PASS=0
FAIL=0
SKIP=0
FAILED_PKGS=()

while IFS= read -r line || [ -n "$line" ]; do
  [[ -z "$line" || "$line" == \#* ]] && continue

  PKG="$line"
  printf "  %-40s" "$PKG"

  RESULT=$($PIP install "$PKG" -q 2>&1)
  STATUS=$?

  if [ $STATUS -eq 0 ]; then
    if echo "$RESULT" | grep -q "already satisfied"; then
      echo "[ already installed ]"
      ((SKIP++))
    else
      echo "[ installed ]"
      ((PASS++))
    fi
  else
    echo "[ FAILED ]"
    echo "    └─ $(echo "$RESULT" | tail -3)"
    FAILED_PKGS+=("$PKG")
    ((FAIL++))
  fi

done < requirements.txt

echo ""
echo "========================================="
echo " SUMMARY"
echo "========================================="
printf "  %-25s %d\n" "Newly installed:"  "$PASS"
printf "  %-25s %d\n" "Already installed:" "$SKIP"
printf "  %-25s %d\n" "Failed:"           "$FAIL"

if [ ${#FAILED_PKGS[@]} -gt 0 ]; then
  echo ""
  echo "  Failed packages:"
  for pkg in "${FAILED_PKGS[@]}"; do
    echo "    - $pkg"
  done
  echo ""
  echo "Setup incomplete. Resolve the above before proceeding."
  exit 1
else
  echo ""
  echo "All packages installed successfully."
fi

## Install D3IL

In [2]:
%%bash
D3IL_ROOT="$HOME/DPCC/dpcc/d3il"
PIP="$HOME/miniconda3/envs/dpcc/bin/pip"

$PIP install -e "$D3IL_ROOT/environments/d3il" -q
$PIP install -e "$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env" -q
echo "✓ D3IL installed"

✓ D3IL installed


## Set PYTHONPATH and MuJoCo renderer

In [ ]:
import sys, os

HOME = os.path.expanduser("~")
D3IL_ROOT   = f"{HOME}/DPCC/dpcc/d3il"
GYM_AV_PATH = f"{D3IL_ROOT}/environments/d3il/envs/gym_avoiding_env"

sys.path.insert(0, D3IL_ROOT)
sys.path.insert(0, GYM_AV_PATH)

os.environ["MUJOCO_GL"]         = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
os.environ["PYTHONPATH"]        = f"{D3IL_ROOT}:{GYM_AV_PATH}:" + os.environ.get("PYTHONPATH", "")
print("✓ Paths and env vars set")

## WSL install OSMesa (If Need)

In [ ]:
# ── Step 1: Install OSMesa system library ────────────────────
sudo apt update
sudo apt install -y libosmesa6 libosmesa6-dev

# ── Step 2: Verify it's now found ────────────────────────────
echo "=== OSMesa after install ==="
ldconfig -p | grep -i osmesa

echo ""
echo "=== Python finds it now? ==="
~/miniconda3/envs/dpcc/bin/python -c "
import ctypes.util
print('OSMesa:', ctypes.util.find_library('OSMesa'))
"

# ── Step 3: Test mujoco imports cleanly ──────────────────────
echo ""
echo "=== MuJoCo test ==="
MUJOCO_GL=osmesa PYOPENGL_PLATFORM=osmesa \
    ~/miniconda3/envs/dpcc/bin/python -c "
import mujoco
print('✓ mujoco', mujoco.__version__)
"

# Here Start in Jupyter Notebook

## Full verification

In [ ]:
%%bash
PYTHON="$HOME/miniconda3/envs/dpcc/bin/python"
D3IL_ROOT="$HOME/DPCC/dpcc/d3il"
GYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"

export PYTHONPATH="$D3IL_ROOT:$GYM_AV"
export MUJOCO_GL="osmesa"
export PYOPENGL_PLATFORM="osmesa"
export LD_PRELOAD="/usr/lib/x86_64-linux-gnu/libstdc++.so.6"
export LD_LIBRARY_PATH="/usr/lib/x86_64-linux-gnu:$LD_LIBRARY_PATH"

$PYTHON - << 'EOF'
import warnings
warnings.filterwarnings("ignore")
import sys

print(f"Python:  {sys.version}")

import torch
print(f"✓ PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

import mujoco
print(f"✓ mujoco {mujoco.__version__}")

from environments.d3il import d3il_sim
print("✓ environments.d3il.d3il_sim")

from environments.d3il.envs.gym_avoiding_env.gym_avoiding.envs.avoiding import ObstacleAvoidanceEnv
env = ObstacleAvoidanceEnv(render=False)
env.start()
obs = env.reset()
print(f"✓ ObstacleAvoidanceEnv OK | obs shape: {obs.shape}")

print("")
print("========================================")
print(" ALL CHECKS PASSED — ready to train!")
print("========================================")
EOF

# Handle Gurobi

In [ ]:
import os
os.environ['GRB_LICENSE_FILE'] = os.path.expanduser('~/DPCC/gurobi.lic')

# Set Up wandb

In [ ]:
!echo $WANDB_MODE

# the WANDB MUST upgrade, other wont work

In [ ]:
~/miniconda3/envs/dpcc/bin/pip install --upgrade wandb

In [ ]:
import wandb
import os

# 1. Ensure any previous run in this process is closed safely
if wandb.run is not None:
    wandb.finish()

# 2. Clean initialization
# Using a 'with' block is often safer as it handles the finish() automatically
run = wandb.init(
    project="DPCC-CPU",
    mode="online",  # Fixed the syntax error here
    reinit=True     # Allows multiple wandb.init() calls in the same notebook/script
)

# 3. Verification
print(f"Current Mode: {wandb.run.settings.mode}")
print(f"Run URL: {run.url}")

# Dataset download

%%bash
ZIP="/content/drive/MyDrive/DPCC/dpcc/d3il/environments/dataset/data/dataset.zip"

echo "=== Top-level folders in zip ==="
unzip -l "$ZIP" | awk '{print $4}' | cut -d'/' -f1 | sort -u

echo ""
echo "=== File count per folder ==="
unzip -l "$ZIP" | awk '{print $4}' | cut -d'/' -f1 | sort | uniq -c | sort -rn

echo ""
echo "=== Total uncompressed size ==="
unzip -l "$ZIP" | tail -1

```
=== Top-level folders in zip ===

----
aligning
avoiding
inserting
Name
pushing
sorting
stacking

=== File count per folder ===
2638779 sorting
1377112 stacking
 392768 aligning
   2002 pushing
    802 inserting
     99 avoiding
      3
      1 Name
      1 ----

=== Total uncompressed size ===
17689115292                     4411562 files

Only Extract the avoiding

In [ ]:
%%bash
DATASET_DIR="$HOME/DPCC/dpcc/d3il/environments/dataset/data"
AVOIDING_DATA="$DATASET_DIR/avoiding/data"
ZIP_FILE="$DATASET_DIR/dataset.zip"

echo "========================================="
echo " D3IL DATASET SETUP"
echo " Source: ALRhub/d3il README.md"
echo " Link: https://drive.google.com/file/d/1SQhbhzV85zf_ltnQ8Cbge2lsSWInxVa8"
echo "========================================="
echo ""

# ── Step 1: Check if avoiding data already extracted ─────────────
if [ -d "$AVOIDING_DATA" ] && [ "$(ls -A $AVOIDING_DATA)" ]; then
    echo "✓ Dataset already extracted at: $AVOIDING_DATA"
    echo "  Files: $(ls $AVOIDING_DATA | wc -l) found"
    echo "  Skipping download and extraction."
    exit 0
fi

# ── Step 2: Check if zip already downloaded ───────────────────────
if [ -f "$ZIP_FILE" ]; then
    echo "✓ Zip already exists: $(du -sh $ZIP_FILE | cut -f1)"
    echo "  Skipping download."
else
    echo "Zip not found. Downloading..."
    mkdir -p "$DATASET_DIR"

    $HOME/miniconda3/envs/dpcc/bin/pip install gdown -q
    echo "✓ gdown ready"

    $HOME/miniconda3/envs/dpcc/bin/python -m gdown \
        "https://drive.google.com/uc?id=1SQhbhzV85zf_ltnQ8Cbge2lsSWInxVa8" \
        -O "$ZIP_FILE"

    if [ ! -f "$ZIP_FILE" ]; then
        echo "✗ Download failed — zip file not found"
        exit 1
    fi
    echo "✓ Downloaded: $(du -sh $ZIP_FILE | cut -f1)"
fi
echo ""

# ── Step 3: Extract ONLY avoiding/ to local disk (fast) ────
echo "Extracting only avoiding/ to local disk first..."
LOCAL="/tmp/avoiding_tmp"
mkdir -p "$LOCAL"
unzip -q "$ZIP_FILE" "avoiding/*" -d "$LOCAL"

if [ $? -ne 0 ]; then
    echo "✗ Extraction failed"
    rm -rf "$LOCAL"
    exit 1
fi
echo "✓ Extracted locally: $(find $LOCAL -type f | wc -l) files | $(du -sh $LOCAL | cut -f1)"
echo ""

# ── Step 4: Copy only avoiding/ to dataset path ───────────────────
echo "Copying avoiding/ to dataset path..."
mkdir -p "$DATASET_DIR/avoiding"
cp -r "$LOCAL/avoiding/." "$DATASET_DIR/avoiding/"
echo "✓ Copied"

# ── Step 5: Cleanup local temp ────────────────────────────────────
rm -rf "$LOCAL"
echo "✓ Local temp cleaned up"
echo ""

# ── Step 6: Final verification ────────────────────────────────────
echo "========================================="
echo " VERIFICATION"
echo "========================================="
if [ -d "$AVOIDING_DATA" ] && [ "$(ls -A $AVOIDING_DATA)" ]; then
    echo "✓ avoiding/data found"
    echo "  Files: $(ls $AVOIDING_DATA | wc -l)"
    echo "  Sample: $(ls $AVOIDING_DATA | head -3)"
    echo ""
    echo "✓ Dataset ready — you can now run the smoke test."
else
    echo "✗ avoiding/data still not found."
    echo "  Actual structure:"
    find "$DATASET_DIR" -maxdepth 3 | sort | head -20
    exit 1
fi

## Minors before running in WSL CPU 

Uninstall triton completely — not needed for CPU

In [ ]:
# Uninstall triton completely — not needed for CPU
~/miniconda3/envs/dpcc/bin/pip uninstall triton -y

echo "✓ triton removed"

# Verify it's gone
~/miniconda3/envs/dpcc/bin/python -c "import triton" 2>&1 \
    && echo "still installed" \
    || echo "✓ triton gone"

Remove Hard Coded GPU CUDA

In [ ]:
# ── Fix 1: arrays.py — change default DEVICE to cpu ─────────
sed -i "s/DEVICE = 'cuda:0'/DEVICE = 'cpu'/" \
    ~/DPCC/dpcc/diffuser/utils/arrays.py

sed -i "s/def batch_to_device(batch, device='cuda:0'):/def batch_to_device(batch, device='cpu'):/" \
    ~/DPCC/dpcc/diffuser/utils/arrays.py

sed -i "s/def to_device(x, device=DEVICE):/def to_device(x, device='cpu'):/" \
    ~/DPCC/dpcc/diffuser/utils/arrays.py

# ── Fix 2: training.py — change default train_device to cpu ──
sed -i "s/train_device='cuda'/train_device='cpu'/" \
    ~/DPCC/dpcc/diffuser/utils/training.py

sed -i "s/pin_memory=True/pin_memory=False/g" \
    ~/DPCC/dpcc/diffuser/utils/training.py

# ── Verify changes ────────────────────────────────────────────
echo "=== arrays.py ==="
grep -n "DEVICE\|batch_to_device\|to_device" \
    ~/DPCC/dpcc/diffuser/utils/arrays.py | head -10

echo ""
echo "=== training.py ==="
grep -n "train_device\|pin_memory" \
    ~/DPCC/dpcc/diffuser/utils/training.py

# Smoke Test Training Cell

%%bash
PYTHON="$HOME/miniconda3/envs/dpcc/bin/python"
PIP="$HOME/miniconda3/envs/dpcc/bin/pip"
DPCC="$HOME/DPCC/dpcc"
D3IL_ROOT="$DPCC/d3il"
GYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"

export PYTHONPATH="$DPCC:$D3IL_ROOT:$GYM_AV"
export MUJOCO_GL="osmesa"
export WANDB_MODE="offline"
export MPLBACKEND="agg"

cd $DPCC

# Force reinstall gymnasium and minari to resolve potential import issues
$PIP install --force-reinstall gymnasium minari -q

$PYTHON - << 'EOF'
import matplotlib
matplotlib.use('agg')   # force non-interactive backend before any other import

import torch
import diffuser.utils as utils

exp = 'avoiding-d3il'
seeds = [5]

class Parser(utils.Parser):
    dataset: str = exp
    config: str = 'config.' + exp

for seed in seeds:
    args = Parser().parse_args(experiment='diffusion', seed=seed)
    torch.manual_seed(args.seed)

    # Smoke test overrides
    args.n_train_steps     = 10
    args.n_steps_per_epoch = 10
    args.batch_size        = 2
    args.device            = 'cpu'

    print(f"Smoke test: seed={seed} | steps={args.n_train_steps} | batch={args.batch_size}")

    dataset_config = utils.Config(
        args.loader,
        savepath=(args.savepath, 'dataset_config.pkl'),
        env=args.dataset,
        horizon=args.horizon,
        normalizer=args.normalizer,
        preprocess_fns=args.preprocess_fns,
        use_padding=args.use_padding,
        max_path_length=args.max_path_length,
        include_returns=args.include_returns,
        returns_scale=args.max_path_length,
        discount=args.discount,
    )
    dataset = dataset_config()
    observation_dim = dataset.observation_dim
    action_dim      = dataset.action_dim
    goal_dim        = dataset.goal_dim

    model_config = utils.Config(
        args.model,
        savepath=(args.savepath, 'model_config.pkl'),
        horizon=args.horizon,
        transition_dim=observation_dim + action_dim,
        cond_dim=observation_dim,
        dim_mults=args.dim_mults,
        returns_condition=args.returns_condition,
        dim=args.dim,
        condition_dropout=args.condition_dropout,
        device=args.device,
    )

    diffusion_config = utils.Config(
        args.diffusion,
        savepath=(args.savepath, 'diffusion_config.pkl'),
        horizon=args.horizon,
        observation_dim=observation_dim,
        action_dim=action_dim,
        goal_dim=goal_dim,
        n_timesteps=args.n_diffusion_steps,
        loss_type=args.loss_type,
        clip_denoised=args.clip_denoised,
        predict_epsilon=args.predict_epsilon,
        action_weight=args.action_weight,
        loss_discount=args.loss_discount,
        returns_condition=args.returns_condition,
        condition_guidance_w=args.condition_guidance_w,
        device=args.device,
    )

    trainer_config = utils.Config(
        utils.Trainer,
        savepath=(args.savepath, 'trainer_config.pkl'),
        train_test_split=args.train_test_split,
        ema_decay=args.ema_decay,
        n_train_steps=args.n_train_steps,
        n_steps_per_epoch=args.n_steps_per_epoch,
        train_batch_size=args.batch_size,
        train_lr=args.learning_rate,
        gradient_accumulate_every=args.gradient_accumulate_every,
        results_folder=args.savepath,
    )

    model     = model_config()
    diffusion = diffusion_config(model)
    trainer   = trainer_config(diffusion, dataset)

    trainer.train()
    print(f"\n✓ Smoke test PASSED — seed {seed} completed {args.n_train_steps} steps")

print("\n========================================")
print(" SMOKE TEST COMPLETE — DPCC code works!")
print("========================================")
EOF

Before Real Training - change device cuda → cpu

In [ ]:
# ── Fix config file: change device cuda → cpu ────────────────
sed -i "s/'device': 'cuda'/'device': 'cpu'/" \
    ~/DPCC/dpcc/config/avoiding-d3il.py

# Also fix any 'cuda:0' variant
sed -i "s/'device': 'cuda:0'/'device': 'cpu'/" \
    ~/DPCC/dpcc/config/avoiding-d3il.py

# ── Verify ───────────────────────────────────────────────────
echo "=== config/avoiding-d3il.py device line ==="
grep -n "device" ~/DPCC/dpcc/config/avoiding-d3il.py

 # Train (using the conda env Python)

In [ ]:
%%bash
DPCC="$HOME/DPCC/dpcc"
D3IL_ROOT="$DPCC/d3il"
GYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"

cd $DPCC

export MUJOCO_GL=osmesa
export PYOPENGL_PLATFORM=osmesa
export WANDB_MODE=offline
export MPLBACKEND=agg
export PYTHONPATH="$DPCC:$D3IL_ROOT:$GYM_AV"
#                   ↑ this was missing — diffuser/ is inside here

~/miniconda3/envs/dpcc/bin/python scripts/train.py

# Evaluate

## CPU Evaluation Updates (Code Changes Outside This Notebook)

The evaluation now runs on CPU because the following source code updates were made outside this notebook:

- `dpcc/projection_eval.py`
  - Added robust local path setup (`sys.path`) for D3IL/gym-avoiding imports.
  - Standardized runtime device handling to CPU via `CPU_DEVICE = 'cpu'`.
  - Kept CPU fallback when loading diffusion checkpoints.

- `dpcc/diffuser/utils/serialization.py`
  - Forced loaded config objects (`model_config`, `diffusion_config`) to honor the requested runtime device.
  - Set trainer runtime device from the same requested device.

- `dpcc/diffuser/utils/training.py`
  - Changed checkpoint loading to `torch.load(..., map_location=self.device)` so CUDA-trained checkpoints load on CPU-only Torch.

- `dpcc/diffuser/sampling/policies.py`
  - Removed hardcoded CUDA tensor placement in policy sampling.
  - Returns and condition tensors now use the model device (CPU-safe).

These changes are persistent in the Python source files, so the Evaluate cell can stay simple and just run `projection_eval.py` on CPU.

In [ ]:
for WSL need

In [ ]:
# ── Downgrade numpy to 1.x ───────────────────────────────────
~/miniconda3/envs/dpcc/bin/pip install "numpy<2" -q

# ── Verify ───────────────────────────────────────────────────
~/miniconda3/envs/dpcc/bin/python -c "import numpy; print('numpy', numpy.__version__)"
# Should show: numpy 1.26.x

# ── Quick test pinocchio loads ────────────────────────────────
~/miniconda3/envs/dpcc/bin/python -c "import pinocchio; print('✓ pinocchio OK')"

then Run

Notice here we use a modified temp version  projection_eval.py under root path! 
only Temp solution. Cuda-cpu issue.

In [ ]:
%%bash
DPCC="$HOME/DPCC/dpcc"
D3IL_ROOT="$DPCC/d3il"
GYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"

cd $DPCC

export CUDA_VISIBLE_DEVICES=""
export MUJOCO_GL=osmesa
export PYOPENGL_PLATFORM=osmesa
export WANDB_MODE=offline
export MPLBACKEND=agg
export PYTHONPATH="$DPCC:$D3IL_ROOT:$GYM_AV"

$HOME/miniconda3/envs/dpcc/bin/python projection_eval.py

echo "Done!"

#  Visualize

In [ ]:
%%bash
DPCC="$HOME/DPCC/dpcc"
D3IL_ROOT="$DPCC/d3il"
GYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"

cd $DPCC

unset MPLBACKEND
export MPLBACKEND=agg
export MUJOCO_GL=osmesa
export PYOPENGL_PLATFORM=osmesa
export PYTHONPATH="$DPCC:$D3IL_ROOT:$GYM_AV"

$HOME/miniconda3/envs/dpcc/bin/python scripts/visualize_data_constraints.py

get the png

In [ ]:
find ~/DPCC/dpcc -name "*.png" -newer ~/DPCC/dpcc/scripts/visualize_data_constraints.py \
    | sort | head -20

In [ ]:
# Load Resut

In [ ]:
%%bash
DPCC="$HOME/DPCC/dpcc"
D3IL_ROOT="$DPCC/d3il"
GYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"

cd $DPCC

unset MPLBACKEND
export MPLBACKEND=agg
export MUJOCO_GL=osmesa
export PYOPENGL_PLATFORM=osmesa
export PYTHONPATH="$DPCC:$D3IL_ROOT:$GYM_AV"

$HOME/miniconda3/envs/dpcc/bin/python scripts/load_results.py

# FM Test Gen3

avoiding-d3il changed into 
        # training
        #'n_steps_per_epoch': 1000,
        #'n_steps_per_epoch': 10,
        'n_steps_per_epoch': 5,
        #'n_train_steps': 1e5,
        'n_train_steps': 10,
        #'batch_size': 8,
        'batch_size': 2,
        'learning_rate': 1e-4,
        #'gradient_accumulate_every': 2,
        'gradient_accumulate_every': 1,
        'ema_decay': 0.995,
        #'train_test_split': 0.9,
        'train_test_split': 1,
        'device': 'cpu',
        'seed': 0,
    },

chang the dpcc/flow_matcher/utils/training.py cuda -> CPU

in ~lin60 utils/training.py
Prevent too small training steps, /0 bug.
#self.save_freq = n_train_steps // 5
self.save_freq = max(1, n_train_steps // 5)

In [14]:
%%bash
DPCC="$HOME/DPCC/dpcc"
D3IL_ROOT="$DPCC/d3il"
GYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"

cd $DPCC

export MUJOCO_GL=osmesa
export PYOPENGL_PLATFORM=osmesa
export WANDB_MODE=offline
export MPLBACKEND=agg
export PYTHONPATH="$DPCC:$D3IL_ROOT:$GYM_AV"

 ~/miniconda3/envs/dpcc/bin/python FM_Unet_v2_test/train_FM_Unet_v2.py --seeds 7

[ train ] Seed source: cli --seeds
[ train ] Training seeds: [7]
[ train ] Saved seed manifest to: logs/avoiding-d3il/flow_matching_unet_v2/H4_K2_Dmodels.diffusion.GaussianDiffusion/seeds_config.json

[utils/config ] Config: <class 'flow_matcher_unet_v2.datasets.sequence.SequenceDataset'>
    discount: 0.99
    env: avoiding-d3il
    horizon: 4
    include_returns: True
    max_path_length: 150
    normalizer: LimitsNormalizer
    preprocess_fns: []
    returns_scale: 150
    use_padding: True

[ utils/config ] Saved config to: logs/avoiding-d3il/flow_matching_unet_v2/H4_K2_Dmodels.diffusion.GaussianDiffusion/7/dataset_config_resume_1.json



Traceback (most recent call last):
  File "/home/liu/DPCC/dpcc/FM_Unet_v2_test/train_FM_Unet_v2.py", line 224, in <module>
    dataset = dataset_config()
  File "/home/liu/DPCC/dpcc/flow_matcher_unet_v2/utils/config.py", line 93, in __call__
    instance = self._class(*args, **kwargs, **self._dict)
  File "/home/liu/DPCC/dpcc/flow_matcher_unet_v2/datasets/sequence.py", line 34, in __init__
    for i, episode in enumerate(itr):
  File "/home/liu/DPCC/dpcc/flow_matcher_unet_v2/datasets/d4rl.py", line 143, in sequence_dataset
    env_state = pickle.load(f)
_pickle.UnpicklingError: invalid load key, '['.


CalledProcessError: Command 'b'DPCC="$HOME/DPCC/dpcc"\nD3IL_ROOT="$DPCC/d3il"\nGYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"\n\ncd $DPCC\n\nexport MUJOCO_GL=osmesa\nexport PYOPENGL_PLATFORM=osmesa\nexport WANDB_MODE=offline\nexport MPLBACKEND=agg\nexport PYTHONPATH="$DPCC:$D3IL_ROOT:$GYM_AV"\n\n ~/miniconda3/envs/dpcc/bin/python FM_Unet_v2_test/train_FM_Unet_v2.py --seeds 7\n'' returned non-zero exit status 1.

# Gen 4 Code Test 

In [16]:
%%bash
DPCC="$HOME/DPCC/dpcc"
D3IL_ROOT="$DPCC/d3il"
GYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"

cd $DPCC

export MUJOCO_GL=osmesa
export PYOPENGL_PLATFORM=osmesa

export WANDB_MODE=offline
export MPLBACKEND=agg
export PYTHONPATH="$DPCC:$D3IL_ROOT:$GYM_AV"

 ~/miniconda3/envs/dpcc/bin/python FM_v3_avoiding_visual_test/train_FM_v3_avoiding_visual.py --seeds 6

[ train ] Seed source: cli --seeds
[ train ] Training seeds: [6]
[ train ] Saved seed manifest to: logs/avoiding-d3il-visual/flow_matching_v3_avoiding_visual/H8_K20_Dmodels.diffusion.GaussianDiffusion/seeds_config.json

[utils/config ] Config: <class 'flow_matcher_v3_avoiding_visual.datasets.sequence.SequenceDataset'>
    discount: 0.99
    env: avoiding-d3il-visual
    horizon: 8
    include_returns: True
    max_path_length: 150
    normalizer: LimitsNormalizer
    preprocess_fns: []
    returns_scale: 150
    use_padding: True

[ utils/config ] Saved config to: logs/avoiding-d3il-visual/flow_matching_v3_avoiding_visual/H8_K20_Dmodels.diffusion.GaussianDiffusion/6/dataset_config_resume_10.json

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)

[utils/config ] Config: <class 'flow_matcher_v3_avoiding_visual.models.unet1d_tempo

Epoch 0:   0%|          | 0/5 [00:00<?, ?it/s]/home/liu/miniconda3/envs/dpcc/lib/python3.10/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Epoch 0: 100%|██████████| 5/5 [00:02<00:00,  1.87it/s, a0_loss=1.89, diffusion_loss=1.47, loss=0.735, lr=5e-7, step=4]                                   


Initial test loss:   0.7180, a0 loss:   1.3061
FM_v3 training completed.


Try the standard (after cpu pathed Eval) for Gen4

In [19]:
%%bash
DPCC="$HOME/DPCC/dpcc"
D3IL_ROOT="$DPCC/d3il"
GYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"

cd $DPCC

export MUJOCO_GL=osmesa
export PYOPENGL_PLATFORM=osmesa

export WANDB_MODE=offline
export MPLBACKEND=agg
export PYTHONPATH="$DPCC:$D3IL_ROOT:$GYM_AV"

 ~/miniconda3/envs/dpcc/bin/python FM_v3_avoiding_visual_test/eval_FM_v3_avoiding_visual.py

pybullet build time: Nov 28 2023 23:45:17



[ utils/serialization ] Loading model from logs/avoiding-d3il-visual/flow_matching_v3_avoiding_visual/H8_K20_Dmodels.diffusion.GaussianDiffusion/6

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)
[ utils/training ] Restored loss history from checkpoint at step 0
Final IK error (78 iterations):  7.285862274065073e-06
Final IK error (0 iterations):  7.285862274065073e-06
------------------------Running avoiding-d3il-visual - top-right-hard - dpcc-r (6)----------------------------
Success rate: 0.0
Constraints satisfied: 0.0
Success rate (goal and constraints): 0.0
Avg number of steps: 0.00 +- 0.00
Avg number of constraint violations: 0.00 +- 0.00
Avg total violation: 0.000 +- 0.000
Average computation time per step: 0.279
------------------------Running avoiding-d3il-visual - top-right-hard - dpcc-r-tightened (6)---------------

Traceback (most recent call last):
  File "/home/liu/DPCC/dpcc/FM_v3_avoiding_visual_test/eval_FM_v3_avoiding_visual.py", line 180, in <module>
    action, samples = policy(conditions={0: obs}, batch_size=args.batch_size, horizon=args.horizon, disable_projection=disable_projection)
  File "/home/liu/DPCC/dpcc/flow_matcher_v3_avoiding_visual/sampling/policies.py", line 52, in __call__
    samples, infos = self.model(conditions, returns=returns, projector=projector, constraints=constraints, horizon=horizon, **self.sample_kwargs)
  File "/home/liu/miniconda3/envs/dpcc/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/home/liu/miniconda3/envs/dpcc/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
  File "/home/liu/DPCC/dpcc/flow_matcher_v3_avoiding_visual/models/diffusion.py", line 296, in forward
    return self.conditional

Error while terminating subprocess (pid=220341): 


TypeError: %d format: a real number is required, not NoneType

Use the version of /eval_FM_CPU.py!

In [14]:
%%bash
DPCC="$HOME/DPCC/dpcc"
D3IL_ROOT="$DPCC/d3il"
GYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"

cd $DPCC

unset MPLBACKEND
export MPLBACKEND=agg
export MUJOCO_GL=osmesa
export PYOPENGL_PLATFORM=osmesa
export PYTHONPATH="$DPCC:$D3IL_ROOT:$GYM_AV"

 ~/miniconda3/envs/dpcc/bin/python FM_Unet_v2_test/eval_FM_CPU.py

pybullet build time: Nov 28 2023 23:45:17


------------------------Loading Model (Seed 7)----------------------------

[ utils/serialization ] Loading model from logs/avoiding-d3il/flow_matching_unet_v2/H4_K2_Dmodels.diffusion.GaussianDiffusion/7

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)
[ utils/training ] Restored loss history from checkpoint at step 1
Final IK error (78 iterations):  7.285862274065073e-06
Final IK error (0 iterations):  7.285862274065073e-06
------------------------Running avoiding-d3il - top-right-hard - dpcc-r (7)----------------------------
Success rate: 0.0
Constraints satisfied: 0.0
Success rate (goal and constraints): 0.0
Avg number of steps: 0.00 +- 0.00
Avg number of constraint violations: 0.00 +- 0.00
Avg total violation: 0.000 +- 0.000
Average computation time per step: 0.069
------------------------Running avoiding-d3il - top-right-

load_result! Require the training seed finished

In [15]:
%%bash
DPCC="$HOME/DPCC/dpcc"
D3IL_ROOT="$DPCC/d3il"
GYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"

cd $DPCC

unset MPLBACKEND
export MPLBACKEND=agg
export MUJOCO_GL=osmesa
export PYOPENGL_PLATFORM=osmesa
export PYTHONPATH="$DPCC:$D3IL_ROOT:$GYM_AV"

 ~/miniconda3/envs/dpcc/bin/python FM_Unet_v2_test/load_results_FM_Unet_v2.py

/home/liu/DPCC/dpcc/FM_Unet_v2_test/load_results_FM_Unet_v2.py:137: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/home/liu/DPCC/dpcc/FM_Unet_v2_test/load_results_FM_Unet_v2.py:155: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/home/liu/DPCC/dpcc/FM_Unet_v2_test/load_results_FM_Unet_v2.py:137: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/home/liu/DPCC/dpcc/FM_Unet_v2_test/load_results_FM_Unet_v2.py:155: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


------------------ Variant: dpcc-r ------------------
Success rate (goal): 0.00
Success rate (goal + constraints): 0.00
Success rate (constraints): 0.00
Average steps: 0.00 +- 0.00
Average violations: 0.00 +- 0.00
Average total violations: 0.000 +- 0.000
Average time: 0.07 +- 0.00
$0.0 \pm 0.0$ & $0.00$ & $0.00$ & $0.0 \pm 0.0$ \\
------------------ Variant: dpcc-r-tightened ------------------
Success rate (goal): 0.00
Success rate (goal + constraints): 0.00
Success rate (constraints): 0.00
Average steps: 0.00 +- 0.00
Average violations: 0.00 +- 0.00
Average total violations: 0.001 +- 0.000
Average time: 0.14 +- 0.09
$0.0 \pm 0.0$ & $0.00$ & $0.00$ & $0.0 \pm 0.0$ \\
------------------ Variant: dpcc-c ------------------
Success rate (goal): 0.00
Success rate (goal + constraints): 0.00
Success rate (constraints): 0.00
Average steps: 0.00 +- 0.00
Average violations: 0.00 +- 0.00
Average total violations: 0.000 +- 0.000
Average time: 0.07 +- 0.01
$0.0 \pm 0.0$ & $0.00$ & $0.00$ & $0.0 \pm

# Load the Plan Npz Logs

In [ ]:
import numpy as np

# Path to your specific file
file_path = 'dpcc/logs/avoiding-d3il/plans/H8_K20_Dmodels.diffusion.GaussianDiffusion/6/results/halfspace_top-right-hard/dpcc-t.npz'

# allow_pickle=True is MANDATORY for 'object' dtypes in npz
# Using a context manager ('with') is safer for file I/O
with np.load(file_path, allow_pickle=True) as data:
    
    # Force NumPy to print the entire array instead of truncating with "..."
    np.set_printoptions(threshold=np.inf, precision=6, suppress=True)
    
    print(f"{'Key':<25} | {'Shape':<15} | {'Data Type':<10}")
    print("=" * 60)

    for key in data.files:
        array = data[key]
        
        # Header for each entry
        print(f"{key:<25} | {str(array.shape):<15} | {str(array.dtype):<10}")
        
        # Logic to handle potential object arrays or nested dicts
        if array.dtype == 'object':
            print("--- Object Content ---")
            print(array.tolist()) # Converting to list often reveals the internal structure
        else:
            print(array)
            
        print("-" * 60)

# load the training logs

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

# ── Fallback Unpickler ──
class FallbackUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if 'flow_matcher' in module:
            return type(name, (), {})
        return super().find_class(module, name)

# ── Target directory ──
target_dir = 'dpcc/logs/avoiding-d3il/flow_matching/H8_K20_Dmodels.diffusion.GaussianDiffusion/6'

path_parts = target_dir.rstrip('/').split('/')
identifier = '__'.join(path_parts[3:]).replace('Dmodels.diffusion.', '').replace('.', '_')
print(f"Run identifier: {identifier}\n")

all_files = [
    f for f in os.listdir(target_dir)
    if os.path.isfile(os.path.join(target_dir, f)) and ':' not in f
]

print(f"Found {len(all_files)} files in: {target_dir}\n")

for file_name in all_files:
    path = os.path.join(target_dir, file_name)
    print(f"{'='*30} FILE: {file_name} {'='*30}")

    try:
        # Handle NumPy Archive (.npz)
        if file_name.endswith('.npz'):
            with np.load(path, allow_pickle=True) as data:
                for key in data.files:
                    arr = data[key]
                    print(f"Key: {key:<20} | Shape: {str(arr.shape):<12} | Dtype: {arr.dtype}")
                    if arr.size > 0:
                        print(arr if arr.size < 10 else f"First 5: {arr.flatten()[:5]}...")

        # Handle Pickle Files (.pkl) — always use FallbackUnpickler
        elif file_name.endswith('.pkl'):
            with open(path, 'rb') as f:
                content = FallbackUnpickler(f).load()

            if isinstance(content, dict):
                for k, v in content.items():
                    print(f"{k:<25}: {v}")
            elif hasattr(content, '__dict__'):
                print(f"  Type: {type(content).__name__}")
                for k, v in vars(content).items():
                    print(f"  {k}: {v}")
            elif isinstance(content, (list, np.ndarray)):
                arr_content = np.array(content)
                print(f"Data Type: {type(content)} | Length: {len(arr_content)}")
                print(f"Last Values: {arr_content[-5:]}")
            else:
                print(content)

            # ── Plot loss curves if this is losses.pkl ──
            if file_name == 'losses.pkl' and isinstance(content, dict):
                fig, axes = plt.subplots(1, 2, figsize=(14, 5))

                for key, label, color in [
                    ('training_losses', 'Training Loss', 'tab:blue'),
                    ('test_losses', 'Test Loss', 'tab:orange'),
                ]:
                    if key in content:
                        data = np.array(content[key])
                        axes[0].plot(data[:, 0], data[:, 1], label=label, color=color, alpha=0.8)
                axes[0].set_xlabel('Step')
                axes[0].set_ylabel('Loss')
                axes[0].set_title('Total Loss')
                axes[0].legend()
                axes[0].grid(True, alpha=0.3)

                for key, label, color in [
                    ('training_a0_losses', 'Training a0 Loss', 'tab:blue'),
                    ('test_a0_losses', 'Test a0 Loss', 'tab:orange'),
                ]:
                    if key in content:
                        data = np.array(content[key])
                        axes[1].plot(data[:, 0], data[:, 1], label=label, color=color, alpha=0.8)
                axes[1].set_xlabel('Step')
                axes[1].set_ylabel('Loss')
                axes[1].set_title('a0 Loss (Action Prediction)')
                axes[1].legend()
                axes[1].grid(True, alpha=0.3)

                plt.suptitle(f'Training — {identifier}', fontsize=13, fontweight='bold')
                plt.tight_layout()

                save_name = f'{identifier}__loss_curves.png'
                save_path = os.path.join(target_dir, save_name)
                plt.savefig(save_path, dpi=150)
                plt.show()
                print(f"\n✅ Loss curves saved to {save_path}")

        # Handle PyTorch Weights (.pt)
        elif file_name.endswith('.pt'):
            import torch
            checkpoint = torch.load(path, map_location='cpu')
            if isinstance(checkpoint, dict):
                print(f"Keys in Checkpoint: {list(checkpoint.keys())}")
            else:
                print("Raw Tensor/Object loaded.")

    except Exception as e:
        print(f"FAILED TO LOAD {file_name}: {e}")

    print("-" * 80)

# DPCC target_dir = 'dpcc/logs/avoiding-d3il/diffusion/H8_K20_Dmodels.GaussianDiffusion/5'
